# LeetCode 3729 — Count Distinct Subarrays Divisible by K in Sorted Array bằng Python

- Link: https://leetcode.com/problems/count-distinct-subarrays-divisible-by-k-in-sorted-array/
- Độ khó: **Hard**
- Chủ đề: Tổng tiền tố modulo, đếm cặp cùng số dư, **khử trùng lặp** nhờ mảng đã sắp xếp, đoạn chạy giá trị bằng nhau, số học (`gcd`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numGoodSubarrays(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Đếm **mọi** mảng con có tổng chia hết cho `k` bằng tổng tiền tố modulo (đếm cặp cùng số dư).
2. Chứng minh: trong mảng **đã sắp xếp**, chỉ các mảng con **hằng** (toàn một giá trị) mới có thể trùng nhau về dãy giá trị.
3. Trừ đi phần trùng: một đoạn chạy giá trị `v` dài `m` chứa `m − h + 1` bản sao của `[v]·h` — chỉ giữ 1.
4. Tính phần trừ bằng **công thức đóng** với `g = k / gcd(v, k)`.
5. Kiểm chứng bằng tập (set) các dãy giá trị.

## 2. Đề bài & ví dụ

`nums` đã sắp **không giảm**, `k > 0`. Mảng con **tốt** nếu tổng chia hết cho `k`. Đếm số mảng con tốt **phân biệt theo dãy giá trị** (ví dụ `[1,1,1]` chỉ có 3 mảng con phân biệt: `[1]`, `[1,1]`, `[1,1,1]`).

**Ví dụ 1:** `nums = [1,2,3], k = 3` → `3` (`[1,2]`, `[3]`, `[1,2,3]`).

**Ví dụ 2:** `nums = [2,2,2,2,2,2], k = 6` → `2` (`[2,2,2]`, `[2,2,2,2,2,2]`).

**Ràng buộc:** `n ≤ 10^5`, `nums[i], k ≤ 10^9`.

In [1]:
from typing import List
from collections import Counter
from math import gcd
import random, time

examples = [([1, 2, 3], 3, 3), ([2, 2, 2, 2, 2, 2], 6, 2)]
for a, k, exp in examples:
    print(a, k, "-> kỳ vọng", exp)

[1, 2, 3] 3 -> kỳ vọng 3
[2, 2, 2, 2, 2, 2] 6 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Tổng tiền tố modulo

`sum(nums[i..j−1]) ≡ 0 (mod k)` ⇔ `P[j] ≡ P[i] (mod k)`. Số mảng con tốt (đếm theo **vị trí**) = `Σ_r C(cnt[r], 2)` với `cnt[r]` = số tiền tố có số dư `r`.

In [2]:
def count_positions(nums, k):
    cnt = Counter({0: 1}); s = res = 0
    for x in nums:
        s = (s + x) % k; res += cnt[s]; cnt[s] += 1
    return res

print(count_positions([1, 2, 3], 3), count_positions([2] * 6, 6))

3 5


### 3.2. Vì sao chỉ mảng con hằng bị trùng?

Giả sử mảng con `X` có ít nhất hai giá trị khác nhau `a < b` (trong `X` chúng đứng cạnh nhau tại chỗ chuyển `a → b`). Vì `nums` không giảm, **mọi** `a` trong `nums` đứng trước mọi `b` ⇒ chỗ chuyển `a → b` trong `nums` là **duy nhất** (cuối đoạn chạy `a`, đầu đoạn chạy `b`). Vị trí của `X` bị cố định bởi chỗ chuyển đó ⇒ `X` chỉ xuất hiện **một lần**.

Mảng con hằng `[v]·h` thì nằm trong đoạn chạy `v` (dài `m`) và xuất hiện `m − h + 1` lần.

In [3]:
nums = [1, 2, 2, 2, 5]
occ = Counter(tuple(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1))
print({k: v for k, v in occ.items() if v > 1}, "<- chỉ các dãy hằng lặp lại")

{(2,): 3, (2, 2): 2} <- chỉ các dãy hằng lặp lại


### 3.3. Đếm `h ∈ [1, m]` với `h·v ≡ 0 (mod k)`

`k | h·v` ⇔ `(k / gcd(v, k)) | h`. Đặt `g = k / gcd(v, k)`, các `h` là `g, 2g, …, t·g` với `t = ⌊m / g⌋`.

### 3.4. Minh hoạ: lớp tương đương theo dãy giá trị

Ô dưới gom mọi mảng con (theo vị trí) của một mảng đã sắp thành các lớp theo dãy giá trị, in kích thước từng lớp. Mọi lớp có kích thước > 1 đều là dãy hằng, và kích thước đúng bằng `m − h + 1`.

In [4]:
nums = [1, 1, 2, 3, 3, 3, 4]
classes = {}
for i in range(len(nums)):
    for j in range(i + 1, len(nums) + 1):
        classes.setdefault(tuple(nums[i:j]), []).append((i, j - 1))
big = {k: v for k, v in classes.items() if len(v) > 1}
for key, pos in big.items():
    print(f"{list(key)}: {len(pos)} vị trí {pos}")
print("số lớp:", len(classes), "| số mảng con theo vị trí:", len(nums) * (len(nums) + 1) // 2)

[1]: 2 vị trí [(0, 0), (1, 1)]
[3]: 3 vị trí [(3, 3), (4, 4), (5, 5)]
[3, 3]: 2 vị trí [(3, 4), (4, 5)]
số lớp: 24 | số mảng con theo vị trí: 28


### 3.5. Bảng `g = k / gcd(v, k)`

`g` là độ dài **nhỏ nhất** để dãy hằng `[v]·g` có tổng chia hết cho `k`; mọi độ dài tốt là bội của `g`.

In [5]:
k = 12
for v in [1, 2, 3, 4, 6, 8, 9, 12, 18]:
    g = k // gcd(v, k)
    print(f"v={v:>2}: gcd={gcd(v, k):>2}, g={g:>2}, độ dài tốt: {[h for h in range(1, 25) if h * v % k == 0][:4]} ...")

v= 1: gcd= 1, g=12, độ dài tốt: [12, 24] ...
v= 2: gcd= 2, g= 6, độ dài tốt: [6, 12, 18, 24] ...
v= 3: gcd= 3, g= 4, độ dài tốt: [4, 8, 12, 16] ...
v= 4: gcd= 4, g= 3, độ dài tốt: [3, 6, 9, 12] ...
v= 6: gcd= 6, g= 2, độ dài tốt: [2, 4, 6, 8] ...
v= 8: gcd= 4, g= 3, độ dài tốt: [3, 6, 9, 12] ...
v= 9: gcd= 3, g= 4, độ dài tốt: [4, 8, 12, 16] ...
v=12: gcd=12, g= 1, độ dài tốt: [1, 2, 3, 4] ...
v=18: gcd= 6, g= 2, độ dài tốt: [2, 4, 6, 8] ...


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

1. Đếm tất cả mảng con tốt **theo vị trí**: `total`.
2. Với mỗi đoạn chạy `(v, m)` và mỗi `h` mà `[v]·h` tốt: dãy này được đếm `m − h + 1` lần, nhưng chỉ nên đếm 1 ⇒ trừ `m − h`.

Ví dụ 2: một đoạn chạy `v = 2`, `m = 6`, `k = 6` ⇒ `g = 3`, `h ∈ {3, 6}`. `total = 4 + 1 = 5` (bốn vị trí của `[2,2,2]` và một của cả mảng) — trừ `(6 − 3) + (6 − 6) = 3` ⇒ `2`. ✔

## 5. Cách 1 — Brute Force: tập các dãy giá trị

In [6]:
def brute(nums, k):
    n = len(nums); seen = set()
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if s % k == 0:
                seen.add(tuple(nums[i:j + 1]))
    return len(seen)

for a, k, exp in examples:
    print(brute(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2


**Nút thắt:** `O(n²)` mảng con, mỗi mảng con băm `O(n)` ⇒ `O(n³)`.

## 6. Ý tưởng thuật toán / bước đột phá

`answer = total − Σ_{đoạn chạy (v, m)} Σ_{h = g, 2g, …, tg} (m − h)`, với `g = k / gcd(v, k)`, `t = ⌊m/g⌋`.

Tổng trong có công thức đóng: `Σ_{i=1}^{t} (m − i·g) = t·m − g·t(t+1)/2`.

Toàn bộ `O(n + số đoạn chạy · log)` (do `gcd`).

**Vì sao không cần lo tràn số hay số âm?** `total ≤ n(n+1)/2 ≈ 5·10^9`, phần trừ cho mỗi đoạn chạy `≤ m(m−1)/2`, và kết quả cuối luôn `≥ 0` vì nó đếm số lớp. Python có số nguyên tuỳ ý nên không cần modulo; ở C++/Java cần kiểu 64 bit.

**Trường hợp đặc biệt:** `k = 1` ⇒ `g = 1`, mọi `h` đều tốt ⇒ đáp án là số mảng con **phân biệt** của mảng (xem Bài 3). `v` chia hết cho `k` ⇒ `g = 1` tương tự.

## 7. Các cách tiếp theo

- **Cách 2:** băm dãy bằng **cặp (chỉ số đầu đoạn chạy, …)** — biểu diễn chuẩn mỗi mảng con bằng vị trí xuất hiện **đầu tiên** của nó, đếm `O(n²)` không cần tuple.
- **Cách 3:** tiền tố modulo + trừ trùng lặp bằng vòng lặp trên `h` (`O(n)` vì `Σ m = n`).
- **Cách 4:** tiền tố modulo + công thức đóng (lời giải nộp).

### 7.1. Cách 2 — Chỉ đếm lần xuất hiện đầu tiên

Theo 3.2, mảng con `[i..j]` là **lần xuất hiện đầu tiên** của dãy giá trị của nó ⇔ nó không hằng, **hoặc** nó hằng và bắt đầu ở đầu đoạn chạy. Vậy đếm các `[i..j]` tốt thoả điều kiện đó.

In [7]:
def first_occurrence(nums, k):
    n = len(nums); res = 0
    run_start = [0] * n
    for i in range(1, n):
        run_start[i] = run_start[i - 1] if nums[i] == nums[i - 1] else i
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            constant = nums[j] == nums[i]
            if s % k == 0 and (not constant or run_start[i] == i):
                res += 1
    return res

for a, k, exp in examples:
    print(first_occurrence(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2


### 7.2. Cách 3 — Tiền tố + trừ trùng bằng vòng lặp

In [8]:
def runs(nums):
    out = []; i = 0
    while i < len(nums):
        j = i
        while j < len(nums) and nums[j] == nums[i]: j += 1
        out.append((nums[i], j - i)); i = j
    return out

def prefix_loop(nums, k):
    ans = count_positions(nums, k)
    for v, m in runs(nums):
        for h in range(1, m + 1):
            if h * v % k == 0:
                ans -= m - h
    return ans

for a, k, exp in examples:
    print(prefix_loop(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2


### 7.3. Cách 4 — Công thức đóng (lời giải chính)

In [9]:
def prefix_closed(nums, k):
    ans = count_positions(nums, k)
    for v, m in runs(nums):
        g = k // gcd(v, k)
        t = m // g
        ans -= t * m - g * t * (t + 1) // 2
    return ans

for a, k, exp in examples:
    print(prefix_closed(a, k), "kỳ vọng", exp)

3 kỳ vọng 3
2 kỳ vọng 2


## 8. TRACE từng bước

In [10]:
def trace(nums, k):
    cnt = Counter({0: 1}); s = total = 0
    print("nums =", nums, "| k =", k)
    for i, x in enumerate(nums):
        s = (s + x) % k; add = cnt[s]; total += add; cnt[s] += 1
        print(f"  i={i}, x={x}: P mod k = {s}, đã có {add} tiền tố cùng số dư -> total = {total}")
    ans = total
    for v, m in runs(nums):
        g = k // gcd(v, k); t = m // g; sub = t * m - g * t * (t + 1) // 2
        hs = [g * i for i in range(1, t + 1)]
        print(f"  đoạn chạy v={v}, m={m}: g={g}, h tốt = {hs}, trừ {sub}")
        ans -= sub
    print("đáp án:", ans, "| brute:", brute(nums, k))

trace([2, 2, 2, 2, 2, 2], 6)

nums = [2, 2, 2, 2, 2, 2] | k = 6
  i=0, x=2: P mod k = 2, đã có 0 tiền tố cùng số dư -> total = 0
  i=1, x=2: P mod k = 4, đã có 0 tiền tố cùng số dư -> total = 0
  i=2, x=2: P mod k = 0, đã có 1 tiền tố cùng số dư -> total = 1
  i=3, x=2: P mod k = 2, đã có 1 tiền tố cùng số dư -> total = 2
  i=4, x=2: P mod k = 4, đã có 1 tiền tố cùng số dư -> total = 3
  i=5, x=2: P mod k = 0, đã có 2 tiền tố cùng số dư -> total = 5
  đoạn chạy v=2, m=6: g=3, h tốt = [3, 6], trừ 3
đáp án: 2 | brute: 2


In [11]:
trace([1, 1, 2, 3, 3, 3, 4], 3)

nums = [1, 1, 2, 3, 3, 3, 4] | k = 3
  i=0, x=1: P mod k = 1, đã có 0 tiền tố cùng số dư -> total = 0
  i=1, x=1: P mod k = 2, đã có 0 tiền tố cùng số dư -> total = 0
  i=2, x=2: P mod k = 1, đã có 1 tiền tố cùng số dư -> total = 1
  i=3, x=3: P mod k = 1, đã có 2 tiền tố cùng số dư -> total = 3
  i=4, x=3: P mod k = 1, đã có 3 tiền tố cùng số dư -> total = 6
  i=5, x=3: P mod k = 1, đã có 4 tiền tố cùng số dư -> total = 10
  i=6, x=4: P mod k = 2, đã có 1 tiền tố cùng số dư -> total = 11
  đoạn chạy v=1, m=2: g=3, h tốt = [], trừ 0
  đoạn chạy v=2, m=1: g=3, h tốt = [], trừ 0
  đoạn chạy v=3, m=3: g=1, h tốt = [1, 2, 3], trừ 3
  đoạn chạy v=4, m=1: g=3, h tốt = [], trừ 0
đáp án: 8 | brute: 8


## 9. Code nộp LeetCode

In [12]:
from math import gcd

class Solution:
    def numGoodSubarrays(self, nums: List[int], k: int) -> int:
        cnt = {0: 1}
        s = ans = 0
        for x in nums:
            s = (s + x) % k
            c = cnt.get(s, 0)
            ans += c
            cnt[s] = c + 1
        n = len(nums)
        i = 0
        while i < n:
            j = i
            while j < n and nums[j] == nums[i]:
                j += 1
            m = j - i
            g = k // gcd(nums[i], k)
            t = m // g
            ans -= t * m - g * t * (t + 1) // 2
            i = j
        return ans

sol = Solution()
for a, k, exp in examples:
    got = sol.numGoodSubarrays(a, k)
    print(a, k, "->", got, "✔" if got == exp else "✘")

[1, 2, 3] 3 -> 3 ✔
[2, 2, 2, 2, 2, 2] 6 -> 2 ✔


## 10. Vì sao thuật toán đúng

1. **`total`** đếm mọi cặp `(i, j)` (mảng con theo vị trí) có tổng chia hết `k` — chuẩn tiền tố modulo.
2. **Lớp tương đương:** gom các mảng con theo dãy giá trị. Theo 3.2, lớp của dãy không hằng có đúng 1 phần tử; lớp của dãy hằng `[v]·h` có `m_v − h + 1` phần tử (mọi vị trí bắt đầu trong đoạn chạy `v` sao cho còn đủ `h` phần tử). Mọi phần tử trong một lớp có cùng tổng ⇒ cùng tốt hoặc cùng không.
3. **Số lớp tốt** = `total − Σ_{lớp tốt} (|lớp| − 1)` = `total − Σ_v Σ_{h tốt} (m_v − h)`.
4. `h·v ≡ 0 (mod k)` ⇔ `g | h` với `g = k / gcd(v, k)`; tổng `Σ_{i=1}^{t}(m − ig) = tm − g·t(t+1)/2`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Tập dãy giá trị | `O(n³)` | `O(n³)` |
| 2. Lần xuất hiện đầu | `O(n²)` | `O(n)` |
| 3. Tiền tố + vòng `h` | `O(n)` | `O(n)` |
| 4. Tiền tố + công thức | `O(n + R log k)` | `O(n)` |

## 12. Bộ test

In [13]:
METHODS = [brute, first_occurrence, prefix_loop, prefix_closed, lambda a, k: Solution().numGoodSubarrays(a, k)]
for a, k, exp in examples:
    assert all(f(a, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên
assert Solution().numGoodSubarrays([5], 5) == 1
assert Solution().numGoodSubarrays([5], 3) == 0
assert Solution().numGoodSubarrays([1, 1, 1], 1) == 3                # k = 1: mọi dãy phân biệt đều tốt
assert Solution().numGoodSubarrays([3, 3, 3, 3], 3) == 4
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 ngẫu nhiên: mảng đã sắp, nhiều giá trị lặp
rng = random.Random(3729)
for _ in range(1500):
    n = rng.randint(1, 14)
    a = sorted(rng.randint(1, 5) for _ in range(n)); k = rng.randint(1, 8)
    b = brute(a, k)
    assert all(f(a, k) == b for f in METHODS[1:]), (a, k)
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [16]:
# 12.4 test lớn: đoạn chạy dài và k lớn
rng = random.Random(1)
for name, a, k in [("một đoạn chạy dài", [7] * 10**5, 14),
                   ("ngẫu nhiên đã sắp", sorted(rng.randint(1, 10**9) for _ in range(10**5)), 10**9),
                   ("vài giá trị", sorted(rng.randint(1, 3) for _ in range(10**5)), 6)]:
    t0 = time.time(); x = Solution().numGoodSubarrays(a, k); t1 = time.time()
    print(f"{name}: đáp án {x}, thời gian {t1 - t0:.3f}s, khớp Cách 3: {x == prefix_loop(a, k)}")

một đoạn chạy dài: đáp án 50000, thời gian 0.013s, khớp Cách 3: True


ngẫu nhiên đã sắp: đáp án 1, thời gian 0.068s, khớp Cách 3: True
vài giá trị: đáp án 555571324, thời gian 0.015s, khớp Cách 3: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không khử trùng (đếm theo vị trí)

In [17]:
print("sai:", count_positions([2] * 6, 6), "| đúng:", brute([2] * 6, 6))

sai: 5 | đúng: 2


### Lỗi 2 — Trừ `m − h + 1` thay vì `m − h` (xoá luôn bản gốc)

In [18]:
def wrong_minus_all(nums, k):
    ans = count_positions(nums, k)
    for v, m in runs(nums):
        for h in range(1, m + 1):
            if h * v % k == 0: ans -= m - h + 1                       # SAI
    return ans

print("sai:", wrong_minus_all([2] * 6, 6), "| đúng:", brute([2] * 6, 6))

sai: 0 | đúng: 2


### Lỗi 3 — Nghĩ rằng dãy có hai giá trị cũng có thể lặp, rồi khử trùng bằng tuple (quá chậm)

Cách tuple đúng nhưng `O(n³)` bộ nhớ/thời gian. Quan sát 3.2 cho phép chỉ xử lý dãy hằng. Ô dưới cho thấy với mảng **chưa sắp xếp**, dãy không hằng có thể lặp — điều kiện "đã sắp" là then chốt:

In [19]:
unsorted = [1, 2, 1, 2]
print("mảng chưa sắp: brute =", brute(unsorted, 3), "| công thức (sai vì giả thiết bị vi phạm):", prefix_closed(unsorted, 3))

mảng chưa sắp: brute = 3 | công thức (sai vì giả thiết bị vi phạm): 4


### Lỗi 4 — `g = k // v` thay vì `k // gcd(v, k)`

In [20]:
def wrong_g(nums, k):
    ans = count_positions(nums, k)
    for v, m in runs(nums):
        g = max(1, k // v); t = m // g                                  # SAI
        ans -= t * m - g * t * (t + 1) // 2
    return ans

print("sai:", wrong_g([4] * 5, 6), "| đúng:", brute([4] * 5, 6))

sai: -7 | đúng: 1


### Lỗi 5 — Gộp mọi đoạn chạy cùng giá trị như thể có nhiều đoạn chạy

Trong mảng đã sắp, mỗi giá trị chỉ có **một** đoạn chạy. Nếu tự viết hàm tách đoạn chạy mà lại so sánh với phần tử **đầu mảng** thay vì phần tử **trước đó**, các đoạn bị tách sai và phép trừ sai lệch. Cách an toàn: luôn so `nums[j] == nums[i]` với `i` là đầu đoạn hiện tại (như hàm `runs`), hoặc dùng `itertools.groupby`.

In [21]:
from itertools import groupby
a = [1, 1, 2, 3, 3, 3, 4]
assert runs(a) == [(v, len(list(g))) for v, g in groupby(a)]
print("runs khớp groupby:", runs(a))

runs khớp groupby: [(1, 2), (2, 1), (3, 3), (4, 1)]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng mảng con chia hết `k`" ⇒ tiền tố modulo, đếm cặp cùng số dư.
- "Phân biệt theo **giá trị**" + mảng **đã sắp** ⇒ chỉ dãy hằng trùng lặp.
- "Bao nhiêu `h ≤ m` với `k | h·v`" ⇒ bội của `k / gcd(v, k)`.
- Tổng cấp số cộng cho phần trừ ⇒ công thức đóng.
- Khi đề nói "phân biệt theo dãy giá trị", hãy tự hỏi: hai vị trí khác nhau cho cùng dãy khi nào? Câu trả lời thường đơn giản hơn bạn nghĩ nếu đầu vào có cấu trúc (đã sắp, hoán vị, …).

## 15. Biến thể của bài

1. **Mảng chưa sắp xếp**: phải đếm số **dãy con liên tiếp phân biệt** có tổng chia hết — cần suffix automaton/suffix array + xử lý tổng (khó hơn nhiều).
2. **Tổng bằng đúng `S`**: tiền tố + hash map; khử trùng tương tự.
3. **Đếm theo vị trí** (bản cổ điển LC 974).
4. **Tích chia hết cho `k`** với mảng sắp xếp: đếm theo phân tích thừa số.

In [22]:
# Biến thể 2: số dãy giá trị phân biệt có tổng đúng bằng S (mảng đã sắp)
def distinct_sum_equal(nums, S):
    cnt = Counter({0: 1}); s = ans = 0
    for x in nums:
        s += x; ans += cnt[s - S]; cnt[s] += 1
    for v, m in runs(nums):
        if v > 0 and S % v == 0 and 1 <= S // v <= m:
            ans -= m - S // v
    return ans

def brute_sum(nums, S):
    return len({tuple(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) == S})

rng = random.Random(15)
for _ in range(500):
    a = sorted(rng.randint(1, 4) for _ in range(rng.randint(1, 12))); S = rng.randint(1, 12)
    assert distinct_sum_equal(a, S) == brute_sum(a, S)
print("Biến thể 2 OK")

Biến thể 2 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Tập tuple | Định nghĩa trực tiếp | `O(n³)` | Kiểm chứng |
| 2. Lần đầu xuất hiện | Chuẩn hoá vị trí | `O(n²)` | Hiểu bản chất |
| 3. Tiền tố + vòng `h` | Trừ trùng | `O(n)` | Đủ nhanh |
| 4. Tiền tố + công thức | `gcd`, cấp số cộng | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `numGoodSubarrays([3,3,3], 3)`.

**Bài 2 (tính tay).** `numGoodSubarrays([1,1,2,2], 2)`.

**Bài 3 (điền TODO).** Viết `count_distinct_all(nums)` = số mảng con phân biệt (không cần điều kiện tổng) của mảng **đã sắp**.

**Bài 4 (tự viết lại).** Viết lại lời giải không dùng `gcd`, chỉ với vòng lặp `h` (Cách 3) nhưng dừng sớm khi tìm được `h` nhỏ nhất.

**Bài 5 (mini challenge).** Đếm số dãy giá trị phân biệt có tổng chia hết cho `k` **và** độ dài chẵn.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_distinct_all(nums):
    # TODO: n(n+1)/2 - Σ_đoạn chạy Σ_h (m - h)
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_even_length(nums, k):
    # TODO: số dư theo cặp (P mod k, chỉ số mod 2)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi dãy hằng `[3]·h` đều tốt. (2) Phân biệt: `h = 1, 2, 3`. (3) `3`.

**Bài 2.** (1) Tốt theo vị trí: `[1,1]`, `[2]` (×2), `[2,2]`, `[1,1,2]`? tổng 4 ✔, `[1,2]` tổng 3 ✗, `[1,1,2,2]` tổng 6 ✔, `[1,2,2]` tổng 5 ✗. (2) Khử `[2]` trùng. (3) Đáp án `5`.

**Bài 3.** (1) Mọi dãy hằng `h` trong đoạn chạy dài `m` lặp `m − h + 1` lần. (2) `Σ_h (m − h) = m(m−1)/2`. (3) `n(n+1)/2 − Σ m(m−1)/2`.

**Bài 4.** (1) `h` nhỏ nhất thoả chính là `g`. (2) Tìm bằng vòng lặp `h = 1..m`. (3) Rồi dùng cấp số cộng.

**Bài 5.** (1) Đếm cặp `(P[i], P[j])` cùng số dư và cùng tính chẵn lẻ chỉ số. (2) Khử trùng: chỉ `h` chẵn. (3) Kiểm chứng vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().numGoodSubarrays([3, 3, 3], 3) == 3
assert Solution().numGoodSubarrays([1, 1, 2, 2], 2) == 5 == brute([1, 1, 2, 2], 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3, 4
def count_distinct_all_sol(nums):
    n = len(nums)
    return n * (n + 1) // 2 - sum(m * (m - 1) // 2 for _, m in runs(nums))

def no_gcd(nums, k):
    ans = count_positions(nums, k)
    for v, m in runs(nums):
        g = next((h for h in range(1, m + 1) if h * v % k == 0), None)
        if g is None: continue
        t = m // g; ans -= t * m - g * t * (t + 1) // 2
    return ans

rng = random.Random(4)
for _ in range(500):
    a = sorted(rng.randint(1, 5) for _ in range(rng.randint(1, 12))); k = rng.randint(1, 9)
    assert count_distinct_all_sol(a) == len({tuple(a[i:j]) for i in range(len(a)) for j in range(i + 1, len(a) + 1)})
    assert no_gcd(a, k) == brute(a, k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [27]:
# Bài 5
def count_even_length_sol(nums, k):
    cnt = Counter({(0, 0): 1}); s = ans = 0
    for j, x in enumerate(nums, 1):
        s = (s + x) % k; ans += cnt[(s, j % 2)]; cnt[(s, j % 2)] += 1
    for v, m in runs(nums):
        for h in range(2, m + 1, 2):
            if h * v % k == 0: ans -= m - h
    return ans

def brute_even(nums, k):
    return len({tuple(nums[i:j]) for i in range(len(nums)) for j in range(i + 2, len(nums) + 1, 2) if sum(nums[i:j]) % k == 0})

rng = random.Random(5)
for _ in range(500):
    a = sorted(rng.randint(1, 5) for _ in range(rng.randint(1, 12))); k = rng.randint(1, 9)
    assert count_even_length_sol(a, k) == brute_even(a, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
total = Σ_r C(cnt[P mod k = r], 2)            # theo vị trí, cnt[0] khởi tạo 1
for (v, m) in đoạn chạy:
    g = k // gcd(v, k); t = m // g
    total -= t*m - g*t*(t+1)//2               # mỗi [v]*h tốt chỉ giữ 1 bản
return total
```

**Checklist:**

- [ ] Tiền tố rỗng (`cnt[0] = 1`)?
- [ ] Chỉ khử trùng **dãy hằng** (nhờ mảng đã sắp)?
- [ ] Trừ `m − h`, không phải `m − h + 1`?
- [ ] `g = k / gcd(v, k)`?
- [ ] Đã so với tập tuple?